# RAG from Scratch to Production — Episode 19/20
## PostgreSQL, Alembic and Docker: surviving restarts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/19_postgres_alembic_docker.ipynb)

A RAG app keeps its state in **several places at once**: user accounts and document records in PostgreSQL, vectors in ChromaDB, a keyword index and uploaded files on disk, conversations in memory. Each one has to survive restarts, and they have to keep **agreeing with each other**.

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s bug log has a painful example (#30):

> Retrieval quality quietly degraded over time. ChromaDB's Docker volume was mounted at `/chroma/chroma`, but the image actually wrote its data to `/data`, a path that wasn't mounted. Every container recreation silently lost all vector data, while Postgres kept everything: old documents stayed `status: "completed"` with nothing left to retrieve.

**In this episode:**
1. Map where every piece of the app's state lives.
2. Reproduce bug #30 in miniature: no Docker needed.
3. Build the check that would have caught it.
4. Run database migrations with **Alembic**, including the one that breaks on existing data.

**No API key and no Docker needed.**

## 0. Setup

SQLite stands in for PostgreSQL (same SQLAlchemy and Alembic code), and ChromaDB runs embedded. Same library versions as production.

In [ ]:
%pip install -q "alembic==1.18.5" "SQLAlchemy==2.0.51" "chromadb==1.5.9" sentence-transformers

In [1]:
import shutil
from pathlib import Path
import chromadb
from sqlalchemy import create_engine, text
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

## 1. Where does the state live?

From production's `docker-compose.yml` and code:

| State | Where | Survives an app restart? | Survives `docker compose up --force-recreate`? |
|---|---|---|---|
| Users, documents (records, status) | PostgreSQL, `postgres_data` volume | ✅ | ✅ |
| Vectors | ChromaDB server, `vector_db_data` volume | ✅ | ✅ since the bug #30 fix |
| BM25 keyword index | Pickle file, `bm25_data` volume | ✅ | ✅ |
| Uploaded files, chunk files | `uploads_data`, `chunks_data` volumes | ✅ | ✅ |
| Conversations | Python dictionary in memory | ❌ | ❌ |
| Rate-limit counters | Python dictionary in memory | ❌ | ❌ |

Two rows are lost on every restart **by design** (Episode 18's conversations and rate limits). The interesting part is the rest: **five stores that must stay in sync**. PostgreSQL says a document is "completed"; that's only true if its vectors, BM25 entries and files are all still there.

## 2. Bug #30 in miniature

A "container" is a folder. Inside it, one sub-folder is a mounted **volume** (kept when the container is recreated) and everything else is the container's own **writable layer** (thrown away on recreate). The database file lives on the volume. ChromaDB writes to whatever path it's configured with.

In [2]:
ROOT = Path("containers")

def start(chroma_path):
    """Start the 'containers': database on its volume, ChromaDB at the configured path."""
    (ROOT / "postgres_volume").mkdir(parents=True, exist_ok=True)
    db = create_engine(f"sqlite:///{ROOT / 'postgres_volume' / 'app.db'}")
    vectors = chromadb.PersistentClient(path=str(ROOT / chroma_path)).get_or_create_collection("documents")
    return db, vectors

def recreate_containers():
    """docker compose up --force-recreate: every container's writable layer is thrown away. Volumes are kept."""
    chromadb.api.client.SharedSystemClient.clear_system_cache()   # a new process: nothing cached in memory
    for path in ROOT.iterdir():
        if not path.name.endswith("_volume"):
            shutil.rmtree(path)

DOCUMENTS = {
    1: ["Employees may work remotely up to 2 days per week.", "Wednesday is a mandatory in-office day."],
    2: ["Every employee has an annual learning budget of 1,200 USD.", "It does not roll over."],
}

def upload_and_process(db, vectors):
    """What the background worker does: index the chunks, then mark the document completed."""
    with db.begin() as conn:
        conn.execute(text("CREATE TABLE IF NOT EXISTS documents (id INTEGER PRIMARY KEY, status TEXT)"))
    for doc_id, chunks in DOCUMENTS.items():
        vectors.add(ids=[f"{doc_id}-{i}" for i in range(len(chunks))], documents=chunks,
                    embeddings=embedder.encode(chunks).tolist(), metadatas=[{"document_id": doc_id}] * len(chunks))
        with db.begin() as conn:
            conn.execute(text("INSERT OR REPLACE INTO documents VALUES (:id, 'completed')"), {"id": doc_id})

def ask(vectors, question):
    if vectors.count() == 0:
        return "(no chunks found) → I don't have enough information."
    return vectors.query(query_embeddings=embedder.encode([question]).tolist(), n_results=1)["documents"][0][0]

def report(db, vectors, label):
    with db.connect() as conn:
        completed = conn.execute(text("SELECT COUNT(*) FROM documents WHERE status = 'completed'")).scalar()
    print(f"{label:<28} database: {completed} completed documents   vector store: {vectors.count()} chunks")
    print(f"{'':<28} 'How many remote days?' → {ask(vectors, 'How many remote days per week?')}")

First with the **bug**: ChromaDB is configured to write to a path that isn't the mounted volume (in production, the image's default `/data` instead of the mounted `/chroma/chroma`).

In [3]:
shutil.rmtree(ROOT, ignore_errors=True)
WRONG_PATH = "chroma_container/data"            # inside the container's writable layer

db, vectors = start(WRONG_PATH)
upload_and_process(db, vectors)
report(db, vectors, "Before recreating:")

recreate_containers()
db, vectors = start(WRONG_PATH)
report(db, vectors, "After recreating:")

Before recreating:           database: 2 completed documents   vector store: 4 chunks
                             'How many remote days?' → Employees may work remotely up to 2 days per week.
After recreating:            database: 2 completed documents   vector store: 0 chunks
                             'How many remote days?' → (no chunks found) → I don't have enough information.


Exactly the bug in the log. The database still says **2 completed documents**. The vector store is **empty**. Nothing crashes: the app starts, health checks pass, and every question just gets *"I don't have enough information"*, which looks like a retrieval-quality problem, not a lost-data problem.

And the fix, writing ChromaDB's data to the volume:

In [4]:
chromadb.api.client.SharedSystemClient.clear_system_cache()
shutil.rmtree(ROOT, ignore_errors=True)
RIGHT_PATH = "chroma_volume"                    # the mounted volume

db, vectors = start(RIGHT_PATH)
upload_and_process(db, vectors)
recreate_containers()
db, vectors = start(RIGHT_PATH)
report(db, vectors, "Fixed, after recreating:")

Fixed, after recreating:     database: 2 completed documents   vector store: 4 chunks
                             'How many remote days?' → Employees may work remotely up to 2 days per week.


The production fix also included the right way to **test** it: the bug log notes that a simple restart looked fine even with the broken setup. Only a real `--force-recreate` throws away the container's own files. **Test the failure you're actually protecting against.**

## 3. The check that would have caught it

Bug #30 went unnoticed because each store looked healthy **on its own**. The failure was a **disagreement between stores**. So check that: on startup (or in `/health`), compare what the database says against what the vector store actually holds.

In [5]:
def consistency_check(db, vectors):
    """Every 'completed' document must have chunks in the vector store."""
    with db.connect() as conn:
        completed = [row[0] for row in conn.execute(text("SELECT id FROM documents WHERE status = 'completed'"))]
    problems = []
    for doc_id in completed:
        found = len(vectors.get(where={"document_id": doc_id})["ids"])
        if found == 0:
            problems.append(f"document {doc_id} is 'completed' but has 0 chunks in the vector store")
    return problems


for label, path in [("Broken setup (bug #30)", WRONG_PATH), ("Fixed setup", RIGHT_PATH)]:
    chromadb.api.client.SharedSystemClient.clear_system_cache()
    shutil.rmtree(ROOT, ignore_errors=True)
    db, vectors = start(path)
    upload_and_process(db, vectors)
    recreate_containers()
    db, vectors = start(path)
    problems = consistency_check(db, vectors)
    print(f"── {label}: {'OK' if not problems else 'INCONSISTENT'}")
    for p in problems:
        print(f"   ⚠️ {p}")

── Broken setup (bug #30): INCONSISTENT
   ⚠️ document 1 is 'completed' but has 0 chunks in the vector store
   ⚠️ document 2 is 'completed' but has 0 chunks in the vector store
── Fixed setup: OK


A few lines of code turn a silent, slow-to-notice degradation into a loud error on the first startup after the data was lost. The same check extends to the other stores: BM25 entries per document, and the uploaded file on disk. And a document found inconsistent can be set back to `pending` so the worker **re-indexes it** from the original upload, if that file survived.

## 4. Migrations with Alembic

The database schema changes as the app grows. Production's third migration added a `status` column to `documents`, after real documents already existed. **Alembic** applies schema changes as numbered, reversible steps, in order.

Two existing documents, uploaded before the background worker existed:

In [6]:
from alembic.migration import MigrationContext
from alembic.operations import Operations
import sqlalchemy as sa

def fresh_database():
    engine = create_engine("sqlite:///:memory:")
    with engine.begin() as conn:
        conn.execute(text("CREATE TABLE documents (id INTEGER PRIMARY KEY, filename TEXT NOT NULL)"))
        conn.execute(text("INSERT INTO documents VALUES (1, 'handbook.pdf'), (2, 'benefits.docx')"))
    return engine

**The naive migration:** add a required (`NOT NULL`) `status` column.

In [7]:
engine = fresh_database()
with engine.begin() as conn:
    op = Operations(MigrationContext.configure(conn))
    try:
        op.add_column("documents", sa.Column("status", sa.String(20), nullable=False))
        print("migration succeeded")
    except Exception as e:
        print("MIGRATION FAILED:", str(e).split("\n")[0])

MIGRATION FAILED: (sqlite3.OperationalError) Cannot add a NOT NULL column with default value NULL


A required column has to have a value for the rows that **already exist**, and there isn't one. On an empty development database this migration works fine, then fails in production, during a deploy, against real data.

**Production's migration** (`dadef4c76abb`) does it in steps: add the column with a temporary server default, so existing rows get a value; then remove the default, so new rows must set it explicitly (the app sets `"pending"`). SQLite can't change a column default in place, so Alembic's **batch mode** rebuilds the table for that step; PostgreSQL does it directly.

In [8]:
engine = fresh_database()
with engine.begin() as conn:
    op = Operations(MigrationContext.configure(conn))
    op.add_column("documents", sa.Column("status", sa.String(20), server_default="completed", nullable=False))
    op.add_column("documents", sa.Column("error_message", sa.Text(), nullable=True))
    with op.batch_alter_table("documents") as batch:
        batch.alter_column("status", server_default=None)
    print(conn.execute(text("SELECT * FROM documents")).fetchall())

[(1, 'handbook.pdf', 'completed', None), (2, 'benefits.docx', 'completed', None)]


It works, and look at the value the existing documents got: **`'completed'`**, not `'pending'`. That's a **data decision**, not just a schema one: those documents were uploaded before the worker existed, when processing happened during the upload itself. Marking them `pending` would make the new worker re-process every old document. Marking them `failed` would hide them from users.

**A migration has two parts: the schema change, and what it means for the rows already there.**

## 5. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- ✅ Keeps every durable store on a **named Docker volume**, including ChromaDB since the bug #30 fix (`--path /chroma/chroma`, verified with a real `--force-recreate`).
- ✅ Runs `alembic upgrade head` on every start (`entrypoint.sh`), and its status-column migration handles existing rows correctly.
- ⚠️ Has **no consistency check** between PostgreSQL, ChromaDB, BM25 and the upload files. If they drift apart again (a volume mix-up, a failed restore, a manual cleanup), the only symptom is weaker answers.
- ⚠️ On its free Render deployment, there's no persistent disk at all, so uploads, vectors and the BM25 index are lost on every redeploy, while the managed PostgreSQL keeps the document records. Exactly bug #30's pattern, by design. The project's own `render.yaml` documents this trade-off; Episode 20 looks at deployment.

## Recap

| Lesson | Why |
|---|---|
| Map every store of state, and what survives what | Restart, container recreate and redeploy each lose different things |
| Test the real failure (`--force-recreate`, not a restart) | A restart looked fine even with bug #30 |
| Check that stores **agree**, not just that each is up | Bug #30 was a disagreement, not a crash |
| Migrations need a plan for existing rows | Required columns fail on real data, and default values are data decisions |

**The rule:** "completed" in one database is a claim about **other** systems. Verify it.

**Next, Episode 20: CI/CD and deployment**, the finale: shipping all of this safely, with the tests from Episode 16 running on every change.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)